<a href="https://colab.research.google.com/github/maggiecrowner/DS5001-Final-Project/blob/main/LDA_THETA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#LDA

In [1]:
! git clone https://github.com/maggiecrowner/DS5001-Final-Project.git

Cloning into 'DS5001-Final-Project'...
remote: Enumerating objects: 287, done.
remote: Counting objects: 100% (112/112), done.
remote: Compressing objects: 100% (108/108), done.
remote: Total 287 (delta 64), reused 4 (delta 4), pack-reused 175 (from 1)
Receiving objects: 100% (287/287), 15.07 MiB | 8.12 MiB/s, done.
Resolving deltas: 100% (115/115), done.


In [2]:
!wget -O CORPUS.csv "https://virginia.box.com/shared/static/ijkqovrdgvrmctsdqobymig2p98q9x0m.csv"
!wget -O LIB.csv "https://virginia.box.com/shared/static/fhzudg34je9xls5bfcbi4xdnaiek74rj.csv"

--2026-04-21 02:37:38--  https://virginia.box.com/shared/static/ijkqovrdgvrmctsdqobymig2p98q9x0m.csv
Resolving virginia.box.com (virginia.box.com)... 74.112.186.157, 2620:117:bff0:12d::
Connecting to virginia.box.com (virginia.box.com)|74.112.186.157|:443... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: /public/static/ijkqovrdgvrmctsdqobymig2p98q9x0m.csv [following]
--2026-04-21 02:37:39--  https://virginia.box.com/public/static/ijkqovrdgvrmctsdqobymig2p98q9x0m.csv
Reusing existing connection to virginia.box.com:443.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://virginia.app.box.com/public/static/ijkqovrdgvrmctsdqobymig2p98q9x0m.csv [following]
--2026-04-21 02:37:39--  https://virginia.app.box.com/public/static/ijkqovrdgvrmctsdqobymig2p98q9x0m.csv
Resolving virginia.app.box.com (virginia.app.box.com)... 74.112.186.157, 2620:117:bff0:12d::
Connecting to virginia.app.box.com (virginia.app.box.com)|74.112.186.157|:443.

In [3]:
import pandas as pd
CORPUS = pd.read_csv('CORPUS.csv', delimiter='|', index_col=[0,1,2,3])
LIB = pd.read_csv('LIB.csv', delimiter='|', index_col=0)

In [4]:
DOCS = CORPUS[CORPUS.pos.str.match(r'^NNS?$')]\
    .groupby(['Artist', 'Album', 'Title']).term_str\
    .apply(lambda x: ' '.join(map(str,x)))\
    .to_frame()\
    .rename(columns={'term_str':'doc_str'})

In [5]:
from sklearn.feature_extraction import text
my_stop_words = list(text.ENGLISH_STOP_WORDS)

In [6]:
from sklearn.feature_extraction.text import CountVectorizer
count_engine = CountVectorizer(max_df=.9, min_df=10, stop_words=my_stop_words)
count_model = count_engine.fit_transform(DOCS.doc_str)
TERMS = count_engine.get_feature_names_out()
VOCAB = pd.DataFrame(index=TERMS)
VOCAB.index.name = 'term_str'
COUNTS = pd.DataFrame(count_model.toarray(), index=DOCS.index, columns=TERMS)
COUNTS

act  \
Artist        Album                                 Title                            
Ariana Grande 13 (Original Broadway Cast Recording) Brand New You                0   
              Ariana Grande                         Ariana Grande Tour Guide     0   
                                                    I’m Every Woman/Vogue        0   
                                                    Leave Me Lonely (Reprise)    0   
                                                    Love Me Harder/breathin      0   
...                                                                            ...   
Taylor Swift  “Taylor’s Songs” 2003 Demo            Lucky You                    0   
                                                    My Turn To Be Me             0   
                                                    Never Fade                   0   
                                                    Point of View                0   
                                                    Smokey Black Nights          0   

                                                                               adam  \
Artist        Album                                 Title                             
Ariana Grande 13 (Original Broadway Cast Recording) Brand New You                 0   
              Ariana Grande                         Ariana Grande Tour Guide      0   
                                                    I’m Every Woman/Vogue         0   
                                                    Leave Me Lonely (Reprise)     0   
                                                    Love Me Harder/breathin       0   
...                                                                             ...   
Taylor Swift  “Taylor’s Songs” 2003 Demo            Lucky You                     0   
                                                    My Turn To Be Me              0   
                                                    Never Fade                    0   
                                                    Point of View                 0   
                                                    Smokey Black Nights           0   

                                                                               advice  \
Artist        Album                                 Title                               
Ariana Grande 13 (Original Broadway Cast Recording) Brand New You                   0   
              Ariana Grande                         Ariana Grande Tour Guide        0   
                                                    I’m Every Woman/Vogue           0   
                                                    Leave Me Lonely (Reprise)       0   
                                                    Love Me Harder/breathin         0   
...                                                                               ...   
Taylor Swift  “Taylor’s Songs” 2003 Demo            Lucky You                       0   
                                                    My Turn To Be Me                0   
                                                    Never Fade                      0   
                                                    Point of View                   0   
                                                    Smokey Black Nights             0   

                                                                               affection  \
Artist        Album                                 Title                                  
Ariana Grande 13 (Original Broadway Cast Recording) Brand New You                      0   
              Ariana Grande                         Ariana Grande Tour Guide           0   
                                                    I’m Every Woman/Vogue              0   
                                                    Leave Me Lonely (Reprise)          0   
                                                    Love Me Harder/breathin            0   
...                       

In [7]:
COUNTS.to_csv("/content/DS5001-Final-Project/COUNTS.csv", sep="|", index=True)

In [8]:
import numpy as np
from sklearn.decomposition import LatentDirichletAllocation as LDA

n_topics = 20
max_iter = 100
n_top_terms = 9
TNAMES = [f"T{str(x).zfill(len(str(n_topics)))}" for x in range(n_topics)]
topic_engine = LDA(n_components=20, random_state=42)
topic_model = topic_engine.fit_transform(count_model)

## THETA

In [9]:
THETA = pd.DataFrame(topic_model, index=DOCS.index, columns=TNAMES)
THETA.columns.name = 'topic_id'
THETA.head()

topic_id                                                                            T00  \
Artist        Album                                 Title                                 
Ariana Grande 13 (Original Broadway Cast Recording) Brand New You              0.001250   
              Ariana Grande                         Ariana Grande Tour Guide   0.002778   
                                                    I’m Every Woman/Vogue      0.038265   
                                                    Leave Me Lonely (Reprise)  0.002500   
                                                    Love Me Harder/breathin    0.000685   

topic_id                                                                            T01  \
Artist        Album                                 Title                                 
Ariana Grande 13 (Original Broadway Cast Recording) Brand New You              0.001250   
              Ariana Grande                         Ariana Grande Tour Guide   0.002778   
                                                    I’m Every Woman/Vogue      0.037843   
                                                    Leave Me Lonely (Reprise)  0.002500   
                                                    Love Me Harder/breathin    0.000685   

topic_id                                                                            T02  \
Artist        Album                                 Title                                 
Ariana Grande 13 (Original Broadway Cast Recording) Brand New You              0.001250   
              Ariana Grande                         Ariana Grande Tour Guide   0.002778   
                                                    I’m Every Woman/Vogue      0.000704   
                                                    Leave Me Lonely (Reprise)  0.224402   
                                                    Love Me Harder/breathin    0.000685   

topic_id                                                                            T03  \
Artist        Album                                 Title                                 
Ariana Grande 13 (Original Broadway Cast Recording) Brand New You              0.001250   
              Ariana Grande                         Ariana Grande Tour Guide   0.002778   
                                                    I’m Every Woman/Vogue      0.238869   
                                                    Leave Me Lonely (Reprise)  0.002500   
                                                    Love Me Harder/breathin    0.000685   

topic_id                                                                            T04  \
Artist        Album                                 Title                                 
Ariana Grande 13 (Original Broadway Cast Recording) Brand New You              0.001250   
              Ariana Grande                         Ariana Grande Tour Guide   0.002778   
                                                    I’m Every Woman/Vogue      0.128415   
                                                    Leave Me Lonely (Reprise)  0.207473   
                                                    Love Me Harder/breathin    0.022615   

topic_id                                                                            T05  \
Artist        Album                                 Title                                 
Ariana Grande 13 (Original Broadway Cast Recording) Brand New You              0.001250   
              Ariana Grande                         Ariana Grande Tour Guide   0.002778   
                                                    I’m Every Woman/Vogue      0.000704   
                                                    Leave Me Lonely (Reprise)  0.002500   
                                                    Love Me Harder/breathin    0.000685   

topic_id                                                                            T06  \
Artist        Album                                 Title                        

In [10]:
THETA.to_csv("/content/DS5001-Final-Project/THETA.csv", sep="|", index=True)